In [ ]:
#@title 1. Conectar Colab a Google Drive
#@markdown Accede a Google Drive para cargar modelos, audio y guardar resultados.

from google.colab import drive

drive.mount("/content/drive")

In [ ]:
#@title 2. Instalación de StyleGAN2-ADA + Audio
#@markdown StyleGAN2 se instala temporalmente en Colab.
#@markdown
#@markdown Los modelos, audios y resultados permanecen en Google Drive.
#@markdown Ejecuta esta celda una vez al comenzar cada nueva sesión.

import os
import sys
import shutil
import subprocess
import importlib.util

from pathlib import Path
from google.colab import drive


# ============================================================
# GOOGLE DRIVE
# ============================================================

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")


# ============================================================
# CARPETAS PERSISTENTES
# ============================================================

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/colab-sg2-ada-pytorch"
)

RESULTS_DIR = DRIVE_ROOT / "results"
AUDIO_DIR = DRIVE_ROOT / "audio"
GENERATED_DIR = DRIVE_ROOT / "generated"
PRETRAINED_DIR = DRIVE_ROOT / "pretrained"

for folder in [
    RESULTS_DIR,
    AUDIO_DIR,
    GENERATED_DIR,
    PRETRAINED_DIR
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


# ============================================================
# DEPENDENCIAS
# ============================================================

packages = {
    "ninja": "ninja",
    "imageio": "imageio",
    "imageio_ffmpeg": "imageio-ffmpeg",
    "librosa": "librosa",
    "soundfile": "soundfile",
    "click": "click",
    "requests": "requests",
    "tqdm": "tqdm"
}


missing = []

for module, package in packages.items():

    if importlib.util.find_spec(module) is None:
        missing.append(package)


if missing:

    print(
        "Instalando dependencias:"
    )

    print(
        ", ".join(missing)
    )

    for package in missing:

        print(
            f"Instalando {package}..."
        )

        result = subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                package,
                "--disable-pip-version-check"
            ],
            text=True,
            capture_output=True
        )

        if result.returncode != 0:

            print(
                result.stderr
            )

            raise RuntimeError(
                f"No se pudo instalar {package}"
            )

else:

    print(
        "Dependencias ya disponibles."
    )


# ============================================================
# STYLEGAN2-ADA
# ============================================================

REPO_DIR = Path(
    "/content/stylegan2-ada-pytorch"
)

REPO_URL = (
    "https://github.com/"
    "angelv-salazar/"
    "stylegan2-ada-pytorch.git"
)

BRANCH = "colab-2026"


if REPO_DIR.exists():

    shutil.rmtree(
        REPO_DIR
    )


print()
print(
    "Descargando StyleGAN2-ADA..."
)


subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR)
    ],
    check=True
)


os.chdir(
    REPO_DIR
)


# ============================================================
# VERIFICACIÓN
# ============================================================

import torch
import librosa


branch = subprocess.check_output(
    [
        "git",
        "branch",
        "--show-current"
    ],
    text=True
).strip()


print()
print(
    "----------------------------------------"
)

print(
    "StyleGAN2-ADA instalado correctamente"
)

print(
    "----------------------------------------"
)

print(
    "Repositorio:",
    REPO_DIR
)

print(
    "Rama:",
    branch
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "CUDA:",
    torch.version.cuda
)

print(
    "Librosa:",
    librosa.__version__
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    print()
    print(
        "Todo listo."
    )

else:

    print()
    print(
        "No se detectó GPU."
    )

    print(
        "Activa una GPU desde "
        "Entorno de ejecución > "
        "Cambiar tipo de entorno de ejecución."
    )

In [ ]:
#@title 3. Analizar Audio

import numpy as np
import librosa
import matplotlib.pyplot as plt

from pathlib import Path


# ============================================================
# PARÁMETROS
# ============================================================

audio_path = "/content/drive/MyDrive/colab-sg2-ada-pytorch/audio/nils.mp3" #@param {type:"string"}

# Cantidad de imágenes por segundo que tendrá el video final.
fps_video = 24 #@param {type:"integer"}

# Fragmento del audio que queremos utilizar.
#
# Ejemplo:
# inicio = 30
# fin    = 45
# analiza desde 00:30 hasta 00:45.
#
# fin = 0 significa "hasta el final del audio".
inicio_recorte_audio = 60 #@param {type:"number"}
fin_recorte_audio = 95 #@param {type:"number"}

# Ventana temporal de suavizado expresada en segundos.
#
# 0.00 = reacción inmediata
# 0.05 = rápida
# 0.15 = equilibrada
# 0.30 = suave
# 0.50 = muy suave
suavizado_segundos = 0.15 #@param {type:"number"}


# ============================================================
# COMPROBAR AUDIO
# ============================================================

AUDIO_PATH = Path(
    audio_path
)

if not AUDIO_PATH.exists():

    raise FileNotFoundError(
        f"No se encontró el audio:\n{AUDIO_PATH}"
    )


# ============================================================
# COMPROBAR PARÁMETROS
# ============================================================

if fps_video <= 0:

    raise ValueError(
        "fps_video debe ser mayor que 0."
    )


if inicio_recorte_audio < 0:

    raise ValueError(
        "inicio_recorte_audio no puede ser negativo."
    )


if fin_recorte_audio < 0:

    raise ValueError(
        "fin_recorte_audio no puede ser negativo."
    )


if suavizado_segundos < 0:

    raise ValueError(
        "suavizado_segundos no puede ser negativo."
    )


# ============================================================
# DURACIÓN TOTAL DEL AUDIO
# ============================================================

duracion_audio_original = librosa.get_duration(
    path=str(AUDIO_PATH)
)


if inicio_recorte_audio >= duracion_audio_original:

    raise ValueError(
        "El inicio del recorte está fuera "
        "de la duración del audio."
    )


# ============================================================
# CALCULAR RECORTE
# ============================================================

# fin = 0 significa utilizar hasta el final.

if fin_recorte_audio == 0:

    fin_real = duracion_audio_original

else:

    fin_real = min(
        fin_recorte_audio,
        duracion_audio_original
    )


if fin_real <= inicio_recorte_audio:

    raise ValueError(
        "El fin del recorte debe ser mayor "
        "que el inicio."
    )


duracion_recorte_audio = (
    fin_real
    - inicio_recorte_audio
)


# Estas variables también servirán después
# para colocar exactamente este mismo fragmento
# de audio en el video final.

audio_offset = float(
    inicio_recorte_audio
)

audio_segment_duration = float(
    duracion_recorte_audio
)


# ============================================================
# CARGAR FRAGMENTO DEL AUDIO
# ============================================================

y, sr = librosa.load(
    AUDIO_PATH,
    sr=22050,
    mono=True,
    offset=audio_offset,
    duration=audio_segment_duration
)


# Duración real cargada.
audio_duration = (
    len(y)
    / sr
)


# ============================================================
# FRAMES DEL VIDEO
# ============================================================

total_frames = max(
    2,
    int(
        round(
            audio_duration
            * fps_video
        )
    )
)


video_times = (
    np.arange(total_frames)
    / fps_video
)


# ============================================================
# INFORMACIÓN
# ============================================================

print(
    f"Audio: {AUDIO_PATH.name}"
)

print(
    f"Duración total del audio: "
    f"{duracion_audio_original:.2f} s"
)

print(
    f"Inicio del recorte: "
    f"{inicio_recorte_audio:.2f} s"
)

print(
    f"Fin del recorte: "
    f"{fin_real:.2f} s"
)

print(
    f"Duración utilizada: "
    f"{audio_duration:.2f} s"
)

print(
    f"Sample rate: "
    f"{sr} Hz"
)

print(
    f"FPS del video: "
    f"{fps_video}"
)

print(
    f"Frames del video: "
    f"{total_frames}"
)

print(
    f"Suavizado: "
    f"{suavizado_segundos:.2f} s"
)


# ============================================================
# ESPECTRO
# ============================================================

n_fft = 2048
hop_length = 512


S = np.abs(
    librosa.stft(
        y,
        n_fft=n_fft,
        hop_length=hop_length
    )
)


frequencies = (
    librosa.fft_frequencies(
        sr=sr,
        n_fft=n_fft
    )
)


spectral_times = (
    librosa.frames_to_time(
        np.arange(
            S.shape[1]
        ),
        sr=sr,
        hop_length=hop_length
    )
)


# ============================================================
# BANDAS DE FRECUENCIA
# ============================================================

# Graves:
# 20 Hz - 250 Hz
#
# Medios:
# 250 Hz - 2000 Hz
#
# Agudos:
# 2000 Hz en adelante

low_mask = (
    (frequencies >= 20)
    &
    (frequencies < 250)
)


mid_mask = (
    (frequencies >= 250)
    &
    (frequencies < 2000)
)


high_mask = (
    frequencies >= 2000
)


bass = np.mean(
    S[low_mask],
    axis=0
)


mids = np.mean(
    S[mid_mask],
    axis=0
)


highs = np.mean(
    S[high_mask],
    axis=0
)


# ============================================================
# RMS
# ============================================================

rms = librosa.feature.rms(
    S=S
)[0]


# ============================================================
# ONSETS
# ============================================================

onset = librosa.onset.onset_strength(
    y=y,
    sr=sr,
    hop_length=hop_length
)


onset_times = (
    librosa.frames_to_time(
        np.arange(
            len(onset)
        ),
        sr=sr,
        hop_length=hop_length
    )
)


# ============================================================
# NORMALIZAR FEATURES
# ============================================================

def normalize_feature(x):

    x = np.asarray(
        x,
        dtype=np.float32
    )


    # Comprimir valores extremos.

    x = np.log1p(
        np.maximum(
            x,
            0
        )
    )


    # Evitar que un pico aislado domine toda la señal.

    low = np.percentile(
        x,
        5
    )

    high = np.percentile(
        x,
        95
    )


    if high <= low:

        return np.zeros_like(
            x
        )


    x = (
        (x - low)
        / (high - low)
    )


    return np.clip(
        x,
        0,
        1
    )


bass = normalize_feature(
    bass
)

mids = normalize_feature(
    mids
)

highs = normalize_feature(
    highs
)

rms = normalize_feature(
    rms
)

onset = normalize_feature(
    onset
)


# ============================================================
# CONVERTIR FEATURES AL FPS DEL VIDEO
# ============================================================

def resample_feature(
    feature,
    feature_times
):

    return np.interp(
        video_times,
        feature_times,
        feature
    ).astype(
        np.float32
    )


bass = resample_feature(
    bass,
    spectral_times
)

mids = resample_feature(
    mids,
    spectral_times
)

highs = resample_feature(
    highs,
    spectral_times
)

rms = resample_feature(
    rms,
    spectral_times
)

onset = resample_feature(
    onset,
    onset_times
)


# ============================================================
# SUAVIZADO TEMPORAL
# ============================================================

def smooth_feature(
    feature,
    seconds
):

    # Convertir segundos de suavizado
    # a cantidad de frames.

    window = max(
        1,
        int(
            round(
                seconds
                * fps_video
            )
        )
    )


    if window <= 1:

        return feature


    kernel = (
        np.ones(
            window,
            dtype=np.float32
        )
        / window
    )


    left = (
        window
        // 2
    )


    right = (
        window
        - 1
        - left
    )


    padded = np.pad(
        feature,
        (
            left,
            right
        ),
        mode="edge"
    )


    return np.convolve(
        padded,
        kernel,
        mode="valid"
    ).astype(
        np.float32
    )


# Graves, medios, agudos y RMS
# utilizan el suavizado general.

bass = smooth_feature(
    bass,
    suavizado_segundos
)

mids = smooth_feature(
    mids,
    suavizado_segundos
)

highs = smooth_feature(
    highs,
    suavizado_segundos
)

rms = smooth_feature(
    rms,
    suavizado_segundos
)


# Los onsets conservan una respuesta más rápida.

onset = smooth_feature(
    onset,
    min(
        suavizado_segundos,
        0.06
    )
)


# ============================================================
# GUARDAR FEATURES
# ============================================================

audio_features = {

    "bass": bass,

    "mids": mids,

    "highs": highs,

    "rms": rms,

    "onset": onset
}


# ============================================================
# INFORMACIÓN DEL SUAVIZADO
# ============================================================

ventana_frames = max(
    1,
    int(
        round(
            suavizado_segundos
            * fps_video
        )
    )
)


print()

print(
    "Análisis completado."
)

print(
    "Ventana aproximada de suavizado:",
    f"{ventana_frames} frames"
)


# ============================================================
# VISUALIZACIÓN
# ============================================================

plt.figure(
    figsize=(14, 5)
)


plt.plot(
    video_times,
    bass,
    label="Graves"
)


plt.plot(
    video_times,
    mids,
    label="Medios"
)


plt.plot(
    video_times,
    highs,
    label="Agudos"
)


plt.plot(
    video_times,
    onset,
    label="Onsets",
    alpha=0.8
)


plt.plot(
    video_times,
    rms,
    label="RMS",
    alpha=0.8
)


plt.xlabel(
    "Tiempo dentro del recorte (s)"
)

plt.ylabel(
    "Intensidad normalizada"
)

plt.title(
    "Características extraídas del audio"
)

plt.legend()

plt.show()

In [ ]:
#@title 3B. Calibración Audio-Reactiva · Fluida y Adaptativa

import numpy as np
import librosa
import matplotlib.pyplot as plt
import ipywidgets as widgets

from IPython.display import display, clear_output


# ============================================================
# COMPROBAR CELDA 3
# ============================================================

required_variables = [

    "y",
    "sr",
    "video_times",
    "fps_video",
    "AUDIO_PATH",
    "audio_offset",
    "audio_segment_duration"
]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(

        "Faltan datos de la celda 3:\n"

        + "\n".join(
            missing_variables
        )

        + "\n\nEjecuta primero la celda 3."
    )


# ============================================================
# ANÁLISIS ESPECTRAL
# ============================================================

n_fft = 2048
hop_length = 512


S = np.abs(

    librosa.stft(

        y,

        n_fft=n_fft,

        hop_length=hop_length
    )

) ** 2


frequencies = librosa.fft_frequencies(

    sr=sr,

    n_fft=n_fft
)


spectral_times = librosa.frames_to_time(

    np.arange(
        S.shape[1]
    ),

    sr=sr,

    hop_length=hop_length
)


nyquist = (
    sr / 2
)


# ============================================================
# NORMALIZACIÓN GENERAL
# ============================================================

def normalize_feature(
    feature
):

    feature = np.asarray(

        feature,

        dtype=np.float32
    )


    feature = np.log1p(

        np.maximum(
            feature,
            0
        )
    )


    low = np.percentile(
        feature,
        5
    )


    high = np.percentile(
        feature,
        97
    )


    if high <= low:

        return np.zeros_like(
            feature
        )


    feature = (

        feature
        - low

    ) / (

        high
        - low
    )


    return np.clip(

        feature,

        0,

        1

    ).astype(
        np.float32
    )


# ============================================================
# REMUESTREAR AL FPS DEL VIDEO
# ============================================================

def resample_to_video(
    feature,
    times
):

    return np.interp(

        video_times,

        times,

        feature

    ).astype(
        np.float32
    )


# ============================================================
# EXTRAER BANDA
# ============================================================

def get_band(
    low_frequency,
    high_frequency
):

    high_frequency = min(

        high_frequency,

        nyquist
    )


    mask = (

        (frequencies >= low_frequency)

        &

        (frequencies < high_frequency)
    )


    if not np.any(
        mask
    ):

        return np.zeros(

            len(
                video_times
            ),

            dtype=np.float32
        )


    energy = np.sqrt(

        np.mean(

            S[
                mask,
                :
            ],

            axis=0
        )
    )


    energy = normalize_feature(
        energy
    )


    return resample_to_video(

        energy,

        spectral_times
    )


# ============================================================
# BANDAS PRINCIPALES
# ============================================================

bass_base = get_band(
    20,
    250
)


mids_base = get_band(
    250,
    2000
)


highs_base = get_band(

    2000,

    min(
        10000,
        nyquist
    )
)


# ============================================================
# SEPARACIÓN ARMÓNICA / PERCUSIVA
# ============================================================

y_harmonic, y_percussive = (

    librosa.effects.hpss(
        y
    )
)


# ============================================================
# ONSET PERCUSIVO
# ============================================================

onset_raw = librosa.onset.onset_strength(

    y=y_percussive,

    sr=sr,

    hop_length=hop_length
)


onset_times = librosa.frames_to_time(

    np.arange(
        len(
            onset_raw
        )
    ),

    sr=sr,

    hop_length=hop_length
)


onset_base = normalize_feature(
    onset_raw
)


onset_base = resample_to_video(

    onset_base,

    onset_times
)


# ============================================================
# RMS
# ============================================================

rms_raw = librosa.feature.rms(

    y=y,

    frame_length=n_fft,

    hop_length=hop_length

)[0]


rms_times = librosa.frames_to_time(

    np.arange(
        len(
            rms_raw
        )
    ),

    sr=sr,

    hop_length=hop_length
)


rms_base = normalize_feature(
    rms_raw
)


rms_base = resample_to_video(

    rms_base,

    rms_times
)


# ============================================================
# DENSIDAD DE ATAQUES
# ============================================================

onset_frames_detected = (

    librosa.onset.onset_detect(

        y=y_percussive,

        sr=sr,

        hop_length=hop_length,

        backtrack=False
    )
)


duration_seconds = max(

    float(
        audio_segment_duration
    ),

    0.001
)


ataques_por_segundo = (

    len(
        onset_frames_detected
    )

    / duration_seconds
)


# ============================================================
# MEDIR DINAMISMO
# ============================================================

def dynamics_score(
    feature
):

    feature = np.asarray(

        feature,

        dtype=np.float32
    )


    if len(feature) < 2:

        return 0.0


    variation = np.mean(

        np.abs(

            np.diff(
                feature
            )
        )
    )


    dispersion = np.std(
        feature
    )


    score = (

        dispersion
        * 1.2

        +

        variation
        * 2.5
    )


    return float(

        np.clip(
            score,
            0,
            1
        )
    )


bass_dynamics = dynamics_score(
    bass_base
)


mids_dynamics = dynamics_score(
    mids_base
)


highs_dynamics = dynamics_score(
    highs_base
)


# ============================================================
# FILTRO ATTACK / RELEASE
# ============================================================

def envelope_follow(

    feature,

    attack_ms,

    release_ms
):

    x = np.asarray(

        feature,

        dtype=np.float32
    )


    if len(x) == 0:

        return x


    attack_seconds = max(

        attack_ms
        / 1000.0,

        0.001
    )


    release_seconds = max(

        release_ms
        / 1000.0,

        0.001
    )


    attack_alpha = np.exp(

        -1.0

        / (
            fps_video
            * attack_seconds
        )
    )


    release_alpha = np.exp(

        -1.0

        / (
            fps_video
            * release_seconds
        )
    )


    result = np.zeros_like(
        x
    )


    result[0] = (
        x[0]
    )


    for i in range(
        1,
        len(x)
    ):


        if x[i] > result[i - 1]:

            alpha = (
                attack_alpha
            )

        else:

            alpha = (
                release_alpha
            )


        result[i] = (

            alpha
            * result[i - 1]

            +

            (
                1.0
                - alpha
            )

            * x[i]
        )


    return result.astype(
        np.float32
    )


# ============================================================
# PROMEDIO LOCAL
# ============================================================

def local_average(

    feature,

    seconds
):

    feature = np.asarray(

        feature,

        dtype=np.float32
    )


    window = max(

        1,

        int(
            round(
                seconds
                * fps_video
            )
        )
    )


    # Conviene usar ventanas impares.

    if window % 2 == 0:

        window += 1


    if window <= 1:

        return feature.copy()


    kernel = (

        np.ones(
            window,
            dtype=np.float32
        )

        / window
    )


    pad = (
        window // 2
    )


    padded = np.pad(

        feature,

        (
            pad,
            pad
        ),

        mode="edge"
    )


    return np.convolve(

        padded,

        kernel,

        mode="valid"

    ).astype(
        np.float32
    )


# ============================================================
# REALCE ADAPTATIVO
# ============================================================

def adaptive_enhance(

    feature,

    window_seconds,

    strength,

    maximum_offset
):

    # --------------------------------------------------------
    # Objetivo:
    #
    # detectar variaciones que pueden ser pequeñas
    # en términos absolutos pero importantes dentro
    # de su contexto local.
    #
    # NO sustituimos la señal original.
    #
    # Solo añadimos un realce suave.
    # --------------------------------------------------------

    feature = np.asarray(

        feature,

        dtype=np.float32
    )


    baseline = local_average(

        feature,

        window_seconds
    )


    deviation = (

        feature
        - baseline
    )


    # --------------------------------------------------------
    # Magnitud local de la variación
    # --------------------------------------------------------

    local_scale = local_average(

        np.abs(
            deviation
        ),

        max(
            0.5,
            window_seconds
            * 0.5
        )
    )


    # --------------------------------------------------------
    # Evitar amplificar ruido microscópico.
    # --------------------------------------------------------

    global_variation = float(

        np.percentile(

            np.abs(
                deviation
            ),

            70
        )
    )


    minimum_scale = max(

        0.025,

        global_variation
        * 0.45
    )


    scale = np.maximum(

        local_scale,

        minimum_scale
    )


    # --------------------------------------------------------
    # Normalización adaptativa suave.
    #
    # tanh evita explosiones.
    # --------------------------------------------------------

    normalized_variation = np.tanh(

        deviation

        / (

            scale
            * 2.2

            + 1e-8
        )
    )


    enhancement = (

        normalized_variation

        * maximum_offset

        * strength
    )


    result = (

        feature
        + enhancement
    )


    return np.clip(

        result,

        0,

        1

    ).astype(
        np.float32
    )


# ============================================================
# GANANCIA SUAVE
# ============================================================

def soft_gain(

    feature,

    gain,

    reactivity
):

    amount = (

        np.maximum(
            feature,
            0
        )

        * gain

        * reactivity

        * 1.20
    )


    return np.clip(

        np.tanh(
            amount
        ),

        0,

        1

    ).astype(
        np.float32
    )


# ============================================================
# AUTOAJUSTE
# ============================================================

def calculate_auto_settings():

    # --------------------------------------------------------
    # Música más rápida:
    #
    # usamos MÁS fluidez, no menos.
    #
    # Así muchos eventos pequeños se convierten
    # en movimientos continuos.
    # --------------------------------------------------------

    if ataques_por_segundo >= 6:

        fluidity = 88
        reactivity = 0.95
        percussion_gain = 0.75


    elif ataques_por_segundo >= 4:

        fluidity = 82
        reactivity = 1.00
        percussion_gain = 0.80


    elif ataques_por_segundo >= 2.5:

        fluidity = 76
        reactivity = 1.05
        percussion_gain = 0.85


    elif ataques_por_segundo >= 1.25:

        fluidity = 68
        reactivity = 1.10
        percussion_gain = 0.90


    else:

        fluidity = 60
        reactivity = 1.15
        percussion_gain = 1.00


    # --------------------------------------------------------
    # Una banda que ya es muy cambiante
    # necesita menos ganancia.
    #
    # Una banda plana conserva más espacio
    # para el realce adaptativo.
    # --------------------------------------------------------

    bass_gain = (

        1.05

        - bass_dynamics
        * 0.18
    )


    mids_gain = (

        1.00

        - mids_dynamics
        * 0.18
    )


    highs_gain = (

        0.95

        - highs_dynamics
        * 0.20
    )


    return {

        "bass":
        float(
            np.clip(
                bass_gain,
                0.75,
                1.10
            )
        ),

        "mids":
        float(
            np.clip(
                mids_gain,
                0.75,
                1.10
            )
        ),

        "highs":
        float(
            np.clip(
                highs_gain,
                0.70,
                1.05
            )
        ),

        "percussion":
        float(
            percussion_gain
        ),

        "energy":
        0.90,

        "reactivity":
        float(
            reactivity
        ),

        "fluidity":
        int(
            fluidity
        )
    }


auto_settings = (
    calculate_auto_settings()
)


# ============================================================
# FADERS
# ============================================================

def create_fader(
    value
):

    return widgets.FloatSlider(

        value=value,

        min=0.0,

        max=2.0,

        step=0.05,

        orientation="vertical",

        readout=True,

        readout_format=".2f",

        continuous_update=False,

        layout=widgets.Layout(

            width="85px",

            height="200px"
        )
    )


bass_fader = create_fader(

    auto_settings[
        "bass"
    ]
)


mids_fader = create_fader(

    auto_settings[
        "mids"
    ]
)


highs_fader = create_fader(

    auto_settings[
        "highs"
    ]
)


percussion_fader = create_fader(

    auto_settings[
        "percussion"
    ]
)


energy_fader = create_fader(

    auto_settings[
        "energy"
    ]
)


# ============================================================
# REACTIVIDAD
# ============================================================

reactivity_slider = widgets.FloatSlider(

    value=auto_settings[
        "reactivity"
    ],

    min=0.5,

    max=1.5,

    step=0.05,

    description="Reactividad:",

    continuous_update=False,

    style={
        "description_width":
        "initial"
    },

    layout=widgets.Layout(
        width="430px"
    )
)


# ============================================================
# FLUIDEZ
# ============================================================

fluidity_slider = widgets.IntSlider(

    value=auto_settings[
        "fluidity"
    ],

    min=0,

    max=100,

    step=5,

    description="Fluidez:",

    continuous_update=False,

    style={
        "description_width":
        "initial"
    },

    layout=widgets.Layout(
        width="430px"
    )
)


# ============================================================
# BOTONES
# ============================================================

auto_button = widgets.Button(

    description="Auto fluido",

    button_style="info",

    layout=widgets.Layout(
        width="160px"
    )
)


apply_button = widgets.Button(

    description="Aplicar",

    button_style="success",

    layout=widgets.Layout(
        width="160px"
    )
)


show_curves_checkbox = widgets.Checkbox(

    value=False,

    description="Mostrar curvas"
)


# ============================================================
# SALIDAS
# ============================================================

status_output = widgets.Output()

plot_output = widgets.Output()


# ============================================================
# PERFIL DEL AUDIO
# ============================================================

def detected_profile():

    if ataques_por_segundo >= 6:

        return "muy denso"


    if ataques_por_segundo >= 4:

        return "rápido"


    if ataques_por_segundo >= 2.5:

        return "rítmico rápido"


    if ataques_por_segundo >= 1.25:

        return "rítmico"


    return "lento / continuo"


# ============================================================
# FLUIDEZ -> TIEMPOS
# ============================================================

def fluidity_times():

    f = (

        fluidity_slider.value

        / 100.0
    )


    # --------------------------------------------------------
    # BANDAS
    # --------------------------------------------------------

    band_attack_ms = (

        35

        + f
        * 110
    )


    band_release_ms = (

        120

        + f
        * 380
    )


    # --------------------------------------------------------
    # PERCUSIÓN
    #
    # Sigue siendo más rápida que las bandas,
    # pero sin convertirse en un spike.
    # --------------------------------------------------------

    onset_attack_ms = (

        12

        + f
        * 28
    )


    onset_release_ms = (

        100

        + f
        * 280
    )


    # --------------------------------------------------------
    # ENERGÍA / RMS
    # --------------------------------------------------------

    rms_attack_ms = (

        180

        + f
        * 320
    )


    rms_release_ms = (

        400

        + f
        * 700
    )


    # --------------------------------------------------------
    # VENTANA DE ADAPTACIÓN
    #
    # Con mayor fluidez también analizamos
    # un contexto temporal algo mayor.
    # --------------------------------------------------------

    adaptive_window_seconds = (

        1.25

        + f
        * 1.50
    )


    return {

        "band_attack":
        band_attack_ms,

        "band_release":
        band_release_ms,

        "onset_attack":
        onset_attack_ms,

        "onset_release":
        onset_release_ms,

        "rms_attack":
        rms_attack_ms,

        "rms_release":
        rms_release_ms,

        "adaptive_window":
        adaptive_window_seconds
    }


# ============================================================
# CUÁNTO REALCE NECESITA CADA BANDA
# ============================================================

def adaptive_strength_from_dynamics(
    score
):

    # --------------------------------------------------------
    # Si la banda ya tiene mucha dinámica:
    # poco realce.
    #
    # Si está relativamente plana:
    # más realce.
    # --------------------------------------------------------

    return float(

        np.clip(

            0.60

            - score
            * 0.35,

            0.25,

            0.60
        )
    )


# ============================================================
# APLICAR CALIBRACIÓN
# ============================================================

def apply_calibration(
    b=None
):

    global audio_features
    global audio_features_detalladas


    times = (
        fluidity_times()
    )


    # ========================================================
    # 1. ENVOLVENTES FLUIDAS
    # ========================================================

    bass = envelope_follow(

        bass_base,

        times[
            "band_attack"
        ],

        times[
            "band_release"
        ]
    )


    mids = envelope_follow(

        mids_base,

        times[
            "band_attack"
        ],

        times[
            "band_release"
        ]
    )


    highs = envelope_follow(

        highs_base,

        times[
            "band_attack"
        ]
        * 1.10,

        times[
            "band_release"
        ]
        * 1.10
    )


    # ========================================================
    # 2. REALCE ADAPTATIVO
    # ========================================================
    #
    # Aquí está uno de los cambios principales.
    #
    # Si una parte del audio tiene poca variación
    # absoluta, pero sí variaciones relativas,
    # esas diferencias vuelven a ser visibles.
    #
    # No se crean saltos:
    # maximum_offset limita el realce.
    # ========================================================

    bass = adaptive_enhance(

        bass,

        window_seconds=times[
            "adaptive_window"
        ],

        strength=adaptive_strength_from_dynamics(
            bass_dynamics
        ),

        maximum_offset=0.12
    )


    mids = adaptive_enhance(

        mids,

        window_seconds=times[
            "adaptive_window"
        ],

        strength=adaptive_strength_from_dynamics(
            mids_dynamics
        ),

        maximum_offset=0.13
    )


    highs = adaptive_enhance(

        highs,

        window_seconds=times[
            "adaptive_window"
        ],

        strength=adaptive_strength_from_dynamics(
            highs_dynamics
        ),

        maximum_offset=0.14
    )


    # ========================================================
    # 3. PERCUSIÓN
    # ========================================================

    # --------------------------------------------------------
    # Seguimiento principal, fluido.
    # --------------------------------------------------------

    onset_fluid = envelope_follow(

        onset_base,

        times[
            "onset_attack"
        ],

        times[
            "onset_release"
        ]
    )


    # --------------------------------------------------------
    # Seguimiento absoluto algo más rápido.
    #
    # Esto garantiza que un kick/snare repetitivo
    # siga apareciendo aunque el patrón sea constante.
    # --------------------------------------------------------

    onset_absolute = envelope_follow(

        onset_base,

        max(
            10,
            times[
                "onset_attack"
            ]
            * 0.55
        ),

        max(
            100,
            times[
                "onset_release"
            ]
            * 0.72
        )
    )


    # --------------------------------------------------------
    # Mezcla:
    #
    # 68% comportamiento fluido
    # 32% ataque absoluto
    # --------------------------------------------------------

    onset = (

        onset_fluid
        * 0.68

        +

        onset_absolute
        * 0.32
    )


    # Pequeño realce adaptativo.

    onset = adaptive_enhance(

        onset,

        window_seconds=max(

            1.0,

            times[
                "adaptive_window"
            ]
            * 0.60
        ),

        strength=0.35,

        maximum_offset=0.09
    )


    # ========================================================
    # 4. RMS / ENERGÍA DE LARGO PLAZO
    # ========================================================

    rms = envelope_follow(

        rms_base,

        times[
            "rms_attack"
        ],

        times[
            "rms_release"
        ]
    )


    # ========================================================
    # 5. FADERS
    # ========================================================

    global_strength = (
        reactivity_slider.value
    )


    bass = soft_gain(

        bass,

        bass_fader.value,

        global_strength
    )


    mids = soft_gain(

        mids,

        mids_fader.value,

        global_strength
    )


    highs = soft_gain(

        highs,

        highs_fader.value,

        global_strength
    )


    onset = soft_gain(

        onset,

        percussion_fader.value,

        global_strength
    )


    rms = soft_gain(

        rms,

        energy_fader.value,

        1.0
    )


    # ========================================================
    # FEATURES PARA LA CELDA 5
    # ========================================================

    audio_features = {

        "bass":
        bass.astype(
            np.float32
        ),

        "mids":
        mids.astype(
            np.float32
        ),

        "highs":
        highs.astype(
            np.float32
        ),

        "onset":
        onset.astype(
            np.float32
        ),

        "rms":
        rms.astype(
            np.float32
        )
    }


    # ========================================================
    # FEATURES DETALLADAS
    # ========================================================

    audio_features_detalladas = {

        "graves":
        bass,

        "medios":
        mids,

        "agudos":
        highs,

        "percusion":
        onset,

        "percusion_absoluta":
        onset_absolute,

        "energia":
        rms
    }


    render_status()


# ============================================================
# ESTADO
# ============================================================

def render_status():

    times = (
        fluidity_times()
    )


    with status_output:

        clear_output(
            wait=True
        )


        print(
            "Perfil detectado:",
            detected_profile()
        )


        print(
            "Ataques:",
            f"{ataques_por_segundo:.2f} por segundo"
        )


        print(
            "Fluidez:",
            fluidity_slider.value
        )


        print()

        print(
            "Bandas:",
            f"{times['band_attack']:.0f} ms ataque / "
            f"{times['band_release']:.0f} ms caída"
        )


        print(
            "Percusión:",
            f"{times['onset_attack']:.0f} ms ataque / "
            f"{times['onset_release']:.0f} ms caída"
        )


        print(
            "Energía:",
            f"{times['rms_attack']:.0f} ms ataque / "
            f"{times['rms_release']:.0f} ms caída"
        )


        print()

        print(
            "Adaptación dinámica: automática"
        )


        print(
            "Ventana de adaptación:",
            f"{times['adaptive_window']:.2f} s"
        )


        print()

        print(
            "Calibración aplicada."
        )


    render_plot()


# ============================================================
# GRÁFICO
# ============================================================

def render_plot():

    with plot_output:

        clear_output(
            wait=True
        )


        if not show_curves_checkbox.value:

            return


        if "audio_features" not in globals():

            return


        plt.figure(
            figsize=(13, 4)
        )


        plt.plot(

            video_times,

            audio_features[
                "bass"
            ],

            label="Graves"
        )


        plt.plot(

            video_times,

            audio_features[
                "mids"
            ],

            label="Medios"
        )


        plt.plot(

            video_times,

            audio_features[
                "highs"
            ],

            label="Agudos"
        )


        plt.plot(

            video_times,

            audio_features[
                "onset"
            ],

            label="Percusión"
        )


        plt.plot(

            video_times,

            audio_features[
                "rms"
            ],

            label="Energía",

            alpha=0.75
        )


        plt.ylim(
            -0.05,
            1.05
        )


        plt.xlabel(
            "Tiempo (s)"
        )


        plt.ylabel(
            "Influencia"
        )


        plt.title(
            "Señales audio-reactivas"
        )


        plt.legend(
            ncol=5
        )


        plt.show()


# ============================================================
# AUTO AJUSTAR
# ============================================================

def auto_adjust(
    b=None
):

    settings = (
        calculate_auto_settings()
    )


    bass_fader.value = (
        settings[
            "bass"
        ]
    )


    mids_fader.value = (
        settings[
            "mids"
        ]
    )


    highs_fader.value = (
        settings[
            "highs"
        ]
    )


    percussion_fader.value = (
        settings[
            "percussion"
        ]
    )


    energy_fader.value = (
        settings[
            "energy"
        ]
    )


    reactivity_slider.value = (
        settings[
            "reactivity"
        ]
    )


    fluidity_slider.value = (
        settings[
            "fluidity"
        ]
    )


    apply_calibration()


# ============================================================
# EVENTOS
# ============================================================

auto_button.on_click(
    auto_adjust
)


apply_button.on_click(
    apply_calibration
)


def show_curves_changed(
    change
):

    render_plot()


show_curves_checkbox.observe(

    show_curves_changed,

    names="value"
)


# ============================================================
# INTERFAZ
# ============================================================

def fader_column(

    title,

    subtitle,

    fader
):

    return widgets.VBox(

        [

            widgets.HTML(
                f"<b>{title}</b>"
            ),

            fader,

            widgets.HTML(
                f"<small>{subtitle}</small>"
            )
        ],

        layout=widgets.Layout(

            align_items="center",

            width="105px"
        )
    )


faders_ui = widgets.HBox(

    [

        fader_column(
            "Graves",
            "estructura",
            bass_fader
        ),

        fader_column(
            "Medios",
            "formas",
            mids_fader
        ),

        fader_column(
            "Agudos",
            "detalle",
            highs_fader
        ),

        fader_column(
            "Percusión",
            "ritmo",
            percussion_fader
        ),

        fader_column(
            "Energía",
            "largo plazo",
            energy_fader
        )
    ],

    layout=widgets.Layout(

        gap="10px",

        align_items="flex-start"
    )
)


general_ui = widgets.VBox(

    [

        reactivity_slider,

        fluidity_slider,

        widgets.HBox(

            [

                auto_button,

                apply_button,

                show_curves_checkbox
            ],

            layout=widgets.Layout(
                gap="10px"
            )
        )
    ]
)


ui = widgets.VBox(

    [

        widgets.HTML(
            "<h3>Calibración audio-reactiva</h3>"
        ),

        widgets.HTML(

            "<small>"
            "Movimiento fluido con adaptación automática "
            "a las variaciones del audio."
            "</small>"
        ),

        faders_ui,

        general_ui,

        status_output,

        plot_output
    ]
)


display(
    ui
)


# ============================================================
# AUTOAJUSTE INICIAL
# ============================================================

auto_adjust()

In [ ]:
#@title 4. Cargar Modelo StyleGAN2 y definir Seeds

import os
import re

import numpy as np
import torch

from PIL import Image
from IPython.display import display

import dnnlib
import legacy


# ============================================================
# PARÁMETROS
# ============================================================

network_pkl = "/content/drive/MyDrive/colab-sg2-ada-pytorch/stylegan2-ada-pytorch/results/00096-fungiceno-11gb-gpu-noaug-resumecustom/network-snapshot-000008.pkl" #@param {type:"string"}


# ------------------------------------------------------------
# SEEDS
# ------------------------------------------------------------

# Seed desde el cual comienza el video.
seed_inicio = 356 #@param {type:"integer"}

# Seeds hacia los que irá avanzando la interpolación.
#
# Ejemplos:
#
# 600
# 600,750,920
# 600-605
# 600,700-703,900
#
seeds_interpolacion = "990-993" #@param {type:"string"}


# ------------------------------------------------------------
# STYLEGAN
# ------------------------------------------------------------

truncation = 0.6 #@param {type:"number"}


# ------------------------------------------------------------
# FORMATO
# ------------------------------------------------------------

formato_no_cuadrado = True #@param {type:"boolean"}

ancho = 1920 #@param {type:"integer"}

alto = 1080 #@param {type:"integer"}


# ============================================================
# LEER SEEDS
# ============================================================

def parse_seed_expression(text):

    text = (
        str(text)
        .strip()
        .replace(" ", "")
    )


    if text == "":

        return []


    seeds = []


    # --------------------------------------------------------
    # Permite combinar listas y rangos:
    #
    # 500
    # 500,600,700
    # 500-510
    # 500,600-605,800
    # --------------------------------------------------------

    parts = text.split(",")


    for part in parts:


        if part == "":
            continue


        # ----------------------------------------------------
        # Rango
        # ----------------------------------------------------

        if re.fullmatch(
            r"\d+-\d+",
            part
        ):

            start, end = map(
                int,
                part.split("-")
            )


            if end >= start:

                seeds.extend(
                    range(
                        start,
                        end + 1
                    )
                )

            else:

                seeds.extend(
                    range(
                        start,
                        end - 1,
                        -1
                    )
                )


        # ----------------------------------------------------
        # Seed individual
        # ----------------------------------------------------

        else:

            try:

                seeds.append(
                    int(part)
                )

            except ValueError:

                raise ValueError(
                    "Formato de seeds inválido: "
                    f"{part}"
                )


    return seeds


# ============================================================
# CONSTRUIR SECUENCIA COMPLETA
# ============================================================

interpolation_seed_list = (
    parse_seed_expression(
        seeds_interpolacion
    )
)


if len(
    interpolation_seed_list
) == 0:

    raise ValueError(
        "Debes indicar al menos un "
        "seed de interpolación."
    )


seed_sequence = [

    int(
        seed_inicio
    ),

    *interpolation_seed_list
]


num_seed_segments = (
    len(seed_sequence)
    - 1
)


# ============================================================
# INFORMACIÓN DE LOS SEEDS
# ============================================================

print(
    "Secuencia de seeds:"
)


print(
    " → ".join(
        str(seed)
        for seed
        in seed_sequence
    )
)


print()

print(
    f"Cantidad de seeds: "
    f"{len(seed_sequence)}"
)


print(
    f"Transiciones: "
    f"{num_seed_segments}"
)


# ------------------------------------------------------------
# Si el audio ya fue analizado,
# mostrar cuánto dura aproximadamente cada transición.
# ------------------------------------------------------------

if (
    "audio_segment_duration"
    in globals()
):

    seconds_per_seed_transition = (

        audio_segment_duration
        / num_seed_segments
    )


    print(
        f"Duración del fragmento de audio: "
        f"{audio_segment_duration:.2f} s"
    )


    print(
        "Duración aproximada por transición: "
        f"{seconds_per_seed_transition:.2f} s"
    )


# ============================================================
# DISPOSITIVO
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"
)


# ============================================================
# CARGAR MODELO
# ============================================================

print()

print(
    "Cargando modelo..."
)


with dnnlib.util.open_url(
    network_pkl
) as f:


    # --------------------------------------------------------
    # GENERADOR RECTANGULAR
    # --------------------------------------------------------

    if formato_no_cuadrado:

        data = legacy.load_network_pkl(

            f,

            custom=True,

            size=[
                alto,
                ancho
            ],

            scale_type="symm"
        )


    # --------------------------------------------------------
    # GENERADOR NORMAL
    # --------------------------------------------------------

    else:

        data = legacy.load_network_pkl(
            f
        )


G = data[
    "G_ema"
].to(
    device
)


G.eval()

G.requires_grad_(
    False
)


print(
    "Modelo cargado."
)


print(
    "GPU:",
    torch.cuda.get_device_name(0)
    if torch.cuda.is_available()
    else "CPU"
)


if formato_no_cuadrado:

    print(
        f"Resolución de síntesis: "
        f"{ancho} × {alto}"
    )

    print(
        "Scale type: symm"
    )

else:

    print(
        f"Resolución: "
        f"{G.img_resolution} × "
        f"{G.img_resolution}"
    )


# ============================================================
# LABEL
# ============================================================

if getattr(
    G,
    "c_dim",
    0
) > 0:

    label = torch.zeros(

        [
            1,
            G.c_dim
        ],

        device=device
    )


else:

    label = None


# ============================================================
# SEED -> Z
# ============================================================

def seed_to_z(seed):

    z = (

        np.random

        .RandomState(
            int(seed)
        )

        .randn(
            1,
            G.z_dim
        )

        .astype(
            np.float32
        )
    )


    return torch.from_numpy(
        z
    ).to(
        device=device,
        dtype=torch.float32
    )


# ============================================================
# SEEDS -> W+
# ============================================================

w_sequence_list = []


print()

print(
    "Calculando posiciones en el espacio latente..."
)


with torch.inference_mode():


    for seed in seed_sequence:


        z = seed_to_z(
            seed
        )


        w = G.mapping(

            z,

            label,

            truncation_psi=truncation
        )


        w_sequence_list.append(
            w
        )


# ============================================================
# UNIR TODOS LOS W+
# ============================================================

# Resultado:
#
# w_sequence[0] = seed inicial
# w_sequence[1] = primer seed de interpolación
# w_sequence[2] = segundo seed...
#
# Shape:
#
# [cantidad_seeds, num_ws, w_dim]

w_sequence = torch.cat(

    w_sequence_list,

    dim=0
)


print(
    "Secuencia latente preparada."
)


print(
    "Dimensiones W+:",
    tuple(
        w_sequence.shape
    )
)


# ============================================================
# DIVIDIR W+ EN TRES GRUPOS
# ============================================================

num_ws = (
    w_sequence.shape[1]
)


coarse_end = max(
    1,
    num_ws // 3
)


middle_end = max(

    coarse_end + 1,

    (
        num_ws
        * 2
    )
    // 3
)


print()

print(
    f"Capas W+: "
    f"{num_ws}"
)


print(
    "Graves:",
    f"0:{coarse_end}"
)


print(
    "Medios:",
    f"{coarse_end}:{middle_end}"
)


print(
    "Agudos:",
    f"{middle_end}:{num_ws}"
)


# ============================================================
# VARIABLES DE COMPATIBILIDAD
# ============================================================

# Primer seed de toda la secuencia.

w_base = (
    w_sequence[
        0:1
    ]
)


# Primer destino.
#
# Se conserva temporalmente para mantener
# compatibilidad con código anterior.
#
# La nueva celda de video usará directamente
# toda la variable w_sequence.

w_reaccion = (
    w_sequence[
        1:2
    ]
)


# ============================================================
# PREVIEW DEL SEED INICIAL
# ============================================================

with torch.inference_mode():

    img_tensor = G.synthesis(

        w_base,

        noise_mode="const"
    )


img = (

    (
        img_tensor.clamp(
            -1,
            1
        )
        + 1
    )

    * 127.5
)


img = (

    img

    .permute(
        0,
        2,
        3,
        1
    )

    .cpu()

    .numpy()

    .astype(
        np.uint8
    )
)


base_image = Image.fromarray(

    img[0],

    "RGB"
)


# Reducir solo la vista previa.
# La imagen generada conserva
# su resolución completa.

preview_image = (
    base_image.copy()
)


preview_image.thumbnail(
    (
        500,
        500
    )
)


print()

print(
    f"Imagen inicial · Seed "
    f"{seed_inicio}"
)


display(
    preview_image
)


# ============================================================
# RESUMEN DEL RECORRIDO
# ============================================================

print()

print(
    "Recorrido:"
)


for i in range(
    num_seed_segments
):

    print(

        f"{i + 1}. "

        f"Seed "
        f"{seed_sequence[i]}"

        " → "

        f"Seed "
        f"{seed_sequence[i + 1]}"
    )

In [ ]:
#@title 5. Generar Video Audio-Reactivo · Calibración Visual Automática

import os
import re
import subprocess

import numpy as np
import torch
import imageio.v2 as imageio
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from IPython.display import display, Video


# ============================================================
# SALIDA
# ============================================================

video_path = "/content/drive/MyDrive/colab-sg2-ada-pytorch/generated/audio_reactivo_calibrado.mp4" #@param {type:"string"}

mostrar_diagnostico = False #@param {type:"boolean"}


# ============================================================
# INTENSIDAD VISUAL
# ============================================================
#
# Estos NO vuelven a analizar el audio.
#
# Multiplican el movimiento visual una vez
# calibrado automáticamente para este modelo.
#
# 0.0 = desactivar
# 0.5 = mitad
# 1.0 = calibración automática
# 1.5 = 50 % más
# 2.0 = doble
# ============================================================

intensidad_graves = 1.0 #@param {type:"number"}

intensidad_medios = 0.5 #@param {type:"number"}

intensidad_agudos = 0.5 #@param {type:"number"}

intensidad_noise = 0 #@param {type:"number"}


# ============================================================
# OBJETIVOS VISUALES AUTOMÁTICOS
# ============================================================
#
# Diferencia media de píxel buscada cuando
# una banda está en reacción máxima.
#
# No hace falta modificarlos normalmente.
#
# Los agudos reciben deliberadamente
# un objetivo mayor.
# ============================================================

objetivo_visual_graves = 0.055
objetivo_visual_medios = 0.055
objetivo_visual_agudos = 0.075


# ============================================================
# COMPROBAR CELDAS ANTERIORES
# ============================================================

required_variables = [
    "AUDIO_PATH",
    "audio_offset",
    "audio_segment_duration",
    "fps_video",
    "audio_features",
    "G",
    "w_sequence",
    "seed_sequence",
    "num_seed_segments"
]


missing_variables = [

    variable

    for variable in required_variables

    if variable not in globals()
]


if missing_variables:

    raise RuntimeError(

        "Faltan datos de las celdas anteriores:\n"

        + "\n".join(
            missing_variables
        )

        + "\n\nEjecuta nuevamente "
          "las celdas 3, 3B y 4."
    )


# ============================================================
# DISPOSITIVO
# ============================================================

device = next(
    G.parameters()
).device


# Variables de la celda 4.

label_actual = globals().get(
    "label",
    None
)

truncation_actual = globals().get(
    "truncation",
    1.0
)


# ============================================================
# FEATURES
# ============================================================

required_features = [
    "bass",
    "mids",
    "highs",
    "rms",
    "onset"
]


for feature in required_features:

    if feature not in audio_features:

        raise RuntimeError(
            f"Falta la feature: {feature}"
        )


feature_lengths = [

    len(
        audio_features[
            feature
        ]
    )

    for feature in required_features
]


if len(
    set(
        feature_lengths
    )
) != 1:

    raise RuntimeError(
        "Las features tienen longitudes diferentes."
    )


total_frames_video = (
    feature_lengths[0]
)


if total_frames_video < 2:

    raise RuntimeError(
        "El fragmento de audio es demasiado corto."
    )


# ============================================================
# ARRAYS DE AUDIO
# ============================================================

bass_signal = np.asarray(
    audio_features["bass"],
    dtype=np.float32
)

mids_signal = np.asarray(
    audio_features["mids"],
    dtype=np.float32
)

highs_signal = np.asarray(
    audio_features["highs"],
    dtype=np.float32
)

onset_signal = np.asarray(
    audio_features["onset"],
    dtype=np.float32
)

rms_signal = np.asarray(
    audio_features["rms"],
    dtype=np.float32
)


# ============================================================
# CAPAS W+
# ============================================================

num_ws = int(
    w_sequence.shape[1]
)


if num_ws >= 9:

    coarse_end = 4
    middle_end = 8

else:

    coarse_end = max(
        1,
        num_ws // 4
    )

    middle_end = max(
        coarse_end + 1,
        num_ws // 2
    )

    middle_end = min(
        middle_end,
        num_ws - 1
    )


print(
    "========================================"
)

print(
    "AUDIO-REACTIVO · NUEVA ARQUITECTURA"
)

print(
    "========================================"
)

print()

print(
    "Recorrido:"
)

print(
    " → ".join(
        str(seed)
        for seed in seed_sequence
    )
)

print()

print(
    f"Frames: {total_frames_video}"
)

print(
    f"FPS: {fps_video}"
)

print(
    f"Duración: "
    f"{total_frames_video / fps_video:.2f} s"
)

print()

print(
    "W+:"
)

print(
    f"Graves: 0:{coarse_end}"
)

print(
    f"Medios: {coarse_end}:{middle_end}"
)

print(
    f"Agudos: {middle_end}:{num_ws}"
)


# ============================================================
# SEGMENTOS
# ============================================================

segment_boundaries = np.rint(

    np.linspace(
        0,
        total_frames_video - 1,
        num_seed_segments + 1
    )

).astype(
    int
)


# ============================================================
# FILTRO TEMPORAL
# ============================================================

def smooth_signal(
    signal,
    milliseconds
):

    signal = np.asarray(
        signal,
        dtype=np.float32
    )


    seconds = max(
        milliseconds / 1000.0,
        0.001
    )


    alpha = np.exp(

        -1.0

        / (
            fps_video
            * seconds
        )
    )


    result = np.zeros_like(
        signal
    )

    result[0] = signal[0]


    for i in range(
        1,
        len(signal)
    ):

        result[i] = (

            alpha
            * result[i - 1]

            +

            (
                1.0
                - alpha
            )
            * signal[i]
        )


    return result.astype(
        np.float32
    )


# ============================================================
# NORMALIZAR VARIACIONES
# ============================================================

def normalize_signed(
    signal,
    percentile=85,
    minimum_scale=0.015
):

    signal = np.asarray(
        signal,
        dtype=np.float32
    )


    signal = (

        signal

        - np.median(
            signal
        )
    )


    scale = float(

        np.percentile(
            np.abs(signal),
            percentile
        )
    )


    scale = max(
        scale,
        minimum_scale
    )


    return np.tanh(

        signal
        / scale

    ).astype(
        np.float32
    )


# ============================================================
# DRIVERS AUDIO-REACTIVOS
# ============================================================
#
# La 3B ya proporciona señales fluidas.
#
# Aquí buscamos únicamente la variación
# alrededor de una tendencia más lenta.
# ============================================================

bass_slow = smooth_signal(
    bass_signal,
    700
)

mids_slow = smooth_signal(
    mids_signal,
    520
)

highs_slow = smooth_signal(
    highs_signal,
    350
)

onset_slow = smooth_signal(
    onset_signal,
    250
)


bass_driver_raw = (
    bass_signal
    - bass_slow
)

mids_driver_raw = (
    mids_signal
    - mids_slow
)

highs_driver_raw = (
    highs_signal
    - highs_slow
)

onset_driver_raw = (
    onset_signal
    - onset_slow
)


# ============================================================
# DRIVER DE NOISE
# ============================================================

noise_raw = (

    highs_signal
    * 0.65

    +

    onset_signal
    * 0.35
)


noise_slow = smooth_signal(
    noise_raw,
    600
)


noise_driver = normalize_signed(

    noise_raw
    - noise_slow,

    percentile=80,

    minimum_scale=0.015
)


# ------------------------------------------------------------
# Noise:
#
# driver -1 → aproximadamente 0.35×
# driver  0 → 1×
# driver +1 → aproximadamente 3.5×
# ------------------------------------------------------------

noise_factor_curve = np.exp(

    noise_driver
    * 1.25
    * intensidad_noise
)


noise_factor_curve = np.clip(

    noise_factor_curve,

    0.35,

    3.5

).astype(
    np.float32
)


# ============================================================
# SEED AUTOMÁTICO DETERMINISTA
# ============================================================

def automatic_seed(
    seed_a,
    seed_b,
    salt
):

    value = (

        int(seed_a)
        * 73856093

        +

        int(seed_b)
        * 19349663

        +

        int(salt)
        * 83492791
    )


    return int(
        value
        & 0x7FFFFFFF
    )


# ============================================================
# SEED -> W+
# ============================================================

def automatic_seed_to_w(
    seed
):

    rng = np.random.RandomState(
        int(seed)
    )


    z = rng.randn(

        1,
        G.z_dim

    ).astype(
        np.float32
    )


    z = torch.from_numpy(
        z
    ).to(
        device
    )


    with torch.inference_mode():

        w = G.mapping(

            z,

            label_actual,

            truncation_psi=truncation_actual
        )


    return w


# ============================================================
# EASING DEL RECORRIDO PRINCIPAL
# ============================================================

def base_ease(
    t
):

    # Mezcla de lineal + coseno.
    #
    # Mantiene fluidez sin hacer
    # pausas excesivas sobre cada seed.

    cosine = (

        0.5

        -

        0.5
        * np.cos(
            np.pi * t
        )
    )


    return (

        t * 0.75

        +

        cosine * 0.25
    )


# ============================================================
# DIRECCIÓN SEMÁNTICA LATERAL
# ============================================================

def semantic_direction(

    base_ws,

    w_start,

    w_end,

    w_aux,

    layer_start,

    layer_end
):

    # --------------------------------------------------------
    # Dirección hacia un W real obtenido
    # mediante G.mapping().
    #
    # Ya no usamos ruido gaussiano directamente
    # dentro de W+.
    # --------------------------------------------------------

    direction = (

        w_aux[
            :,
            layer_start:
            layer_end
        ]

        -

        base_ws[
            :,
            layer_start:
            layer_end
        ]
    )


    tangent = (

        w_end[
            :,
            layer_start:
            layer_end
        ]

        -

        w_start[
            :,
            layer_start:
            layer_end
        ]
    )


    # --------------------------------------------------------
    # Proyección GLOBAL del bloque.
    #
    # Quitamos únicamente el movimiento
    # paralelo a A→B.
    # --------------------------------------------------------

    dot = torch.sum(

        direction
        * tangent
    )


    tangent_norm_sq = torch.sum(

        tangent
        * tangent
    )


    if float(
        tangent_norm_sq
    ) > 1e-8:

        direction = (

            direction

            -

            tangent
            * (
                dot
                / tangent_norm_sq
            )
        )


    # --------------------------------------------------------
    # Escala de referencia.
    #
    # MUY IMPORTANTE:
    #
    # ya no dejamos que una banda con poca
    # diferencia A→B produzca una dirección
    # microscópica.
    # --------------------------------------------------------

    full_delta = (

        w_end
        - w_start
    )


    global_rms = torch.sqrt(

        torch.mean(
            full_delta
            * full_delta
        )

        + 1e-8
    )


    layer_rms = torch.sqrt(

        torch.mean(
            tangent
            * tangent
        )

        + 1e-8
    )


    target_rms = torch.maximum(

        layer_rms,

        global_rms
        * 0.65
    )


    direction_rms = torch.sqrt(

        torch.mean(
            direction
            * direction
        )

        + 1e-8
    )


    direction = (

        direction

        / direction_rms

        * target_rms
    )


    return direction


# ============================================================
# SÍNTESIS PARA CALIBRACIÓN
# ============================================================

@torch.no_grad()
def synth_tensor(
    ws
):

    return G.synthesis(

        ws,

        noise_mode="const"
    )


# ============================================================
# DIFERENCIA VISUAL REAL
# ============================================================

def visual_difference(
    image_a,
    image_b
):

    # Imágenes en [-1,1].
    #
    # Dividimos por 2 para expresar
    # aproximadamente la diferencia
    # sobre escala 0–1.

    return float(

        torch.mean(

            torch.abs(
                image_a
                - image_b
            )

        ).item()

        / 2.0
    )


# ============================================================
# CALIBRAR AMPLITUD EN IMAGEN
# ============================================================

def calibrate_direction(

    base_ws,

    base_image,

    direction,

    layer_start,

    layer_end,

    target_difference,

    maximum_gain=2.0,

    iterations=5
):

    # --------------------------------------------------------
    # Búsqueda binaria.
    #
    # No preguntamos:
    #
    # "¿cuánto nos movimos en W?"
    #
    # sino:
    #
    # "¿cuánto CAMBIÓ realmente la imagen?"
    # --------------------------------------------------------

    low = 0.0
    high = maximum_gain


    for _ in range(
        iterations
    ):

        gain = (

            low
            + high

        ) / 2.0


        probe_ws = (
            base_ws.clone()
        )


        probe_ws[
            :,
            layer_start:
            layer_end
        ] += (

            direction
            * gain
        )


        probe_image = synth_tensor(
            probe_ws
        )


        difference = visual_difference(

            base_image,

            probe_image
        )


        if (
            difference
            < target_difference
        ):

            low = gain

        else:

            high = gain


    final_gain = (

        low
        + high

    ) / 2.0


    # Medición final.

    probe_ws = (
        base_ws.clone()
    )


    probe_ws[
        :,
        layer_start:
        layer_end
    ] += (

        direction
        * final_gain
    )


    final_image = synth_tensor(
        probe_ws
    )


    final_difference = visual_difference(

        base_image,

        final_image
    )


    return (
        float(final_gain),
        float(final_difference),
        final_image
    )


# ============================================================
# NOISE_STRENGTH
# ============================================================

noise_parameters = []


for name, parameter in G.named_parameters():

    if not name.endswith(
        "noise_strength"
    ):

        continue


    match = re.search(

        r"(?:^|\.)b(\d+)(?:\.|$)",

        name
    )


    resolution = (

        int(
            match.group(1)
        )

        if match

        else None
    )


    # Priorizamos texturas desde 64 px.

    if (

        resolution is None

        or

        resolution >= 64
    ):

        noise_parameters.append(

            {

                "name":
                name,

                "parameter":
                parameter,

                "base":
                parameter
                .detach()
                .clone(),

                "resolution":
                resolution
            }
        )


# ============================================================
# PREPARAR SEGMENTOS
# ============================================================

segments = []


calibration_preview = None


print()

print(
    "Calibrando automáticamente "
    "la reacción visual..."
)

print(
    "Esto puede tardar unos segundos."
)


for segment_index in range(
    num_seed_segments
):

    start_frame = (
        segment_boundaries[
            segment_index
        ]
    )


    end_frame = (
        segment_boundaries[
            segment_index + 1
        ]
    )


    sl = slice(
        start_frame,
        end_frame + 1
    )


    seed_a = int(
        seed_sequence[
            segment_index
        ]
    )


    seed_b = int(
        seed_sequence[
            segment_index + 1
        ]
    )


    w_start = (

        w_sequence[
            segment_index:
            segment_index + 1
        ]
    )


    w_end = (

        w_sequence[
            segment_index + 1:
            segment_index + 2
        ]
    )


    # ========================================================
    # SEEDS AUXILIARES AUTOMÁTICOS
    # ========================================================

    aux_seed_bass = automatic_seed(
        seed_a,
        seed_b,
        101
    )


    aux_seed_mids = automatic_seed(
        seed_a,
        seed_b,
        202
    )


    aux_seed_highs = automatic_seed(
        seed_a,
        seed_b,
        303
    )


    w_aux_bass = automatic_seed_to_w(
        aux_seed_bass
    )


    w_aux_mids = automatic_seed_to_w(
        aux_seed_mids
    )


    w_aux_highs = automatic_seed_to_w(
        aux_seed_highs
    )


    # ========================================================
    # DRIVERS DEL SEGMENTO
    # ========================================================

    bass_driver = normalize_signed(

        bass_driver_raw[
            sl
        ],

        percentile=82,

        minimum_scale=0.012
    )


    mids_driver = normalize_signed(

        mids_driver_raw[
            sl
        ],

        percentile=82,

        minimum_scale=0.012
    )


    highs_component = normalize_signed(

        highs_driver_raw[
            sl
        ],

        percentile=80,

        minimum_scale=0.010
    )


    onset_component = normalize_signed(

        onset_driver_raw[
            sl
        ],

        percentile=80,

        minimum_scale=0.012
    )


    # Agudos más presentes.

    highs_driver = (

        highs_component
        * 0.70

        +

        onset_component
        * 0.30
    )


    highs_driver = np.clip(

        highs_driver,

        -1.0,

        1.0
    )


    # Medios reciben un poco de onset.

    mids_driver = np.clip(

        mids_driver
        * 0.88

        +

        onset_component
        * 0.12,

        -1.0,

        1.0
    )


    # ========================================================
    # CALIBRACIÓN EN EL CENTRO DEL SEGMENTO
    # ========================================================

    calibration_t = 0.5


    base_ws_mid = torch.lerp(

        w_start,

        w_end,

        calibration_t
    )


    base_image_mid = synth_tensor(
        base_ws_mid
    )


    # --------------------------------------------------------
    # Direcciones semánticas
    # --------------------------------------------------------

    bass_direction_mid = semantic_direction(

        base_ws_mid,

        w_start,

        w_end,

        w_aux_bass,

        0,

        coarse_end
    )


    mids_direction_mid = semantic_direction(

        base_ws_mid,

        w_start,

        w_end,

        w_aux_mids,

        coarse_end,

        middle_end
    )


    highs_direction_mid = semantic_direction(

        base_ws_mid,

        w_start,

        w_end,

        w_aux_highs,

        middle_end,

        num_ws
    )


    # --------------------------------------------------------
    # CALIBRACIÓN VISUAL AUTOMÁTICA
    # --------------------------------------------------------

    bass_gain, bass_diff, bass_probe = (
        calibrate_direction(

            base_ws_mid,

            base_image_mid,

            bass_direction_mid,

            0,

            coarse_end,

            objetivo_visual_graves
        )
    )


    mids_gain, mids_diff, mids_probe = (
        calibrate_direction(

            base_ws_mid,

            base_image_mid,

            mids_direction_mid,

            coarse_end,

            middle_end,

            objetivo_visual_medios
        )
    )


    highs_gain, highs_diff, highs_probe = (
        calibrate_direction(

            base_ws_mid,

            base_image_mid,

            highs_direction_mid,

            middle_end,

            num_ws,

            objetivo_visual_agudos
        )
    )


    # ========================================================
    # GUARDAR SEGMENTO
    # ========================================================

    segments.append(

        {

            "start_frame":
            int(
                start_frame
            ),

            "end_frame":
            int(
                end_frame
            ),

            "seed_a":
            seed_a,

            "seed_b":
            seed_b,

            "w_start":
            w_start,

            "w_end":
            w_end,

            "w_aux_bass":
            w_aux_bass,

            "w_aux_mids":
            w_aux_mids,

            "w_aux_highs":
            w_aux_highs,

            "aux_seed_bass":
            aux_seed_bass,

            "aux_seed_mids":
            aux_seed_mids,

            "aux_seed_highs":
            aux_seed_highs,

            "bass_driver":
            bass_driver,

            "mids_driver":
            mids_driver,

            "highs_driver":
            highs_driver,

            "bass_gain":
            bass_gain,

            "mids_gain":
            mids_gain,

            "highs_gain":
            highs_gain,

            "bass_diff":
            bass_diff,

            "mids_diff":
            mids_diff,

            "highs_diff":
            highs_diff
        }
    )


    print()

    print(
        f"{seed_a} → {seed_b}"
    )


    print(
        "  Graves:"
        f" ganancia {bass_gain:.3f}"
        f" | cambio visual {bass_diff:.3f}"
    )


    print(
        "  Medios:"
        f" ganancia {mids_gain:.3f}"
        f" | cambio visual {mids_diff:.3f}"
    )


    print(
        "  Agudos:"
        f" ganancia {highs_gain:.3f}"
        f" | cambio visual {highs_diff:.3f}"
    )


    # Guardamos primer segmento
    # para diagnóstico visual.

    if segment_index == 0:

        calibration_preview = {

            "base":
            base_image_mid,

            "bass":
            bass_probe,

            "mids":
            mids_probe,

            "highs":
            highs_probe
        }


print()

print(
    "Calibración terminada."
)


# ============================================================
# DIAGNÓSTICO
# ============================================================

if mostrar_diagnostico:

    # ========================================================
    # 1. DRIVERS DEL AUDIO
    # ========================================================

    plt.figure(
        figsize=(14, 5)
    )


    for index, segment in enumerate(
        segments
    ):

        times = (

            np.arange(

                segment[
                    "start_frame"
                ],

                segment[
                    "end_frame"
                ]
                + 1
            )

            / fps_video
        )


        plt.plot(

            times,

            segment[
                "bass_driver"
            ],

            label=(
                "Graves"
                if index == 0
                else None
            )
        )


        plt.plot(

            times,

            segment[
                "mids_driver"
            ],

            label=(
                "Medios"
                if index == 0
                else None
            )
        )


        plt.plot(

            times,

            segment[
                "highs_driver"
            ],

            label=(
                "Agudos + percusión"
                if index == 0
                else None
            )
        )


    plt.axhline(
        0,
        linewidth=1,
        alpha=0.4
    )


    plt.xlabel(
        "Tiempo (s)"
    )


    plt.ylabel(
        "Movimiento lateral -1 ... +1"
    )


    plt.title(
        "Drivers que realmente moverán W+"
    )


    plt.legend()

    plt.show()


    # ========================================================
    # 2. NOISE
    # ========================================================

    plt.figure(
        figsize=(14, 3.5)
    )


    plt.plot(

        np.arange(
            total_frames_video
        )
        / fps_video,

        noise_factor_curve
    )


    plt.axhline(
        1.0,
        linewidth=1,
        alpha=0.4
    )


    plt.xlabel(
        "Tiempo (s)"
    )


    plt.ylabel(
        "Multiplicador"
    )


    plt.title(
        "Noise StyleGAN · Agudos + Percusión"
    )


    plt.show()


    # ========================================================
    # 3. PRUEBA VISUAL REAL
    # ========================================================

    def tensor_to_image(
        tensor
    ):

        image = (

            (
                tensor
                .clamp(
                    -1,
                    1
                )
                + 1
            )

            * 127.5
        )


        return (

            image[0]

            .permute(
                1,
                2,
                0
            )

            .detach()
            .cpu()
            .numpy()

            .astype(
                np.uint8
            )
        )


    preview_names = [
        "Interpolación",
        "Graves +1",
        "Medios +1",
        "Agudos +1"
    ]


    preview_tensors = [
        calibration_preview["base"],
        calibration_preview["bass"],
        calibration_preview["mids"],
        calibration_preview["highs"]
    ]


    plt.figure(
        figsize=(16, 4)
    )


    for i, (
        name,
        tensor
    ) in enumerate(
        zip(
            preview_names,
            preview_tensors
        )
    ):

        plt.subplot(
            1,
            4,
            i + 1
        )


        plt.imshow(
            tensor_to_image(
                tensor
            )
        )


        plt.title(
            name
        )


        plt.axis(
            "off"
        )


    plt.suptitle(
        "Prueba de impacto visual · Primer segmento"
    )


    plt.tight_layout()

    plt.show()


# ============================================================
# NOISE DINÁMICO
# ============================================================

@torch.no_grad()
def synthesize_dynamic(
    ws,
    noise_factor
):

    # --------------------------------------------------------
    # Conservamos noise_mode="const":
    #
    # el patrón espacial no cambia aleatoriamente.
    #
    # Cambia solamente su intensidad.
    # --------------------------------------------------------

    for item in noise_parameters:

        item[
            "parameter"
        ].copy_(

            item[
                "base"
            ]

            * float(
                noise_factor
            )
        )


    try:

        image = G.synthesis(

            ws,

            noise_mode="const"
        )


    finally:

        for item in noise_parameters:

            item[
                "parameter"
            ].copy_(

                item[
                    "base"
                ]
            )


    return image


# ============================================================
# BUSCAR SEGMENTO
# ============================================================

def find_segment(
    frame_index
):

    segment_index = int(

        np.searchsorted(

            segment_boundaries[
                1:
            ],

            frame_index,

            side="left"
        )
    )


    return int(

        np.clip(

            segment_index,

            0,

            num_seed_segments - 1
        )
    )


# ============================================================
# GENERAR FRAME
# ============================================================

def generate_audio_frame(
    frame_index
):

    segment_index = find_segment(
        frame_index
    )


    segment = segments[
        segment_index
    ]


    local_index = (

        frame_index

        - segment[
            "start_frame"
        ]
    )


    segment_length = (

        segment[
            "end_frame"
        ]

        - segment[
            "start_frame"
        ]
    )


    local_t = (

        local_index

        / max(
            1,
            segment_length
        )
    )


    local_t = float(

        np.clip(
            local_t,
            0.0,
            1.0
        )
    )


    # ========================================================
    # INTERPOLACIÓN PRINCIPAL
    # ========================================================

    t = base_ease(
        local_t
    )


    w_start = segment[
        "w_start"
    ]


    w_end = segment[
        "w_end"
    ]


    ws = torch.lerp(

        w_start,

        w_end,

        t
    )


    # ========================================================
    # VENTANA LATERAL
    # ========================================================
    #
    # 0 exactamente en los seeds.
    #
    # 1 hacia el centro.
    #
    # Exponente < 1:
    # mantiene la reacción durante
    # una parte amplia de la transición.
    # ========================================================

    lateral_window = (

        np.sin(
            np.pi
            * local_t
        )

        ** 0.65
    )


    # ========================================================
    # DIRECCIÓN ACTUAL · GRAVES
    # ========================================================

    bass_direction = semantic_direction(

        ws,

        w_start,

        w_end,

        segment[
            "w_aux_bass"
        ],

        0,

        coarse_end
    )


    bass_amount = (

        segment[
            "bass_driver"
        ][
            local_index
        ]

        * segment[
            "bass_gain"
        ]

        * intensidad_graves

        * lateral_window
    )


    ws[
        :,
        :coarse_end
    ] += (

        bass_direction

        * float(
            bass_amount
        )
    )


    # ========================================================
    # DIRECCIÓN ACTUAL · MEDIOS
    # ========================================================

    mids_direction = semantic_direction(

        ws,

        w_start,

        w_end,

        segment[
            "w_aux_mids"
        ],

        coarse_end,

        middle_end
    )


    mids_amount = (

        segment[
            "mids_driver"
        ][
            local_index
        ]

        * segment[
            "mids_gain"
        ]

        * intensidad_medios

        * lateral_window
    )


    ws[
        :,
        coarse_end:
        middle_end
    ] += (

        mids_direction

        * float(
            mids_amount
        )
    )


    # ========================================================
    # DIRECCIÓN ACTUAL · AGUDOS
    # ========================================================

    highs_direction = semantic_direction(

        ws,

        w_start,

        w_end,

        segment[
            "w_aux_highs"
        ],

        middle_end,

        num_ws
    )


    highs_amount = (

        segment[
            "highs_driver"
        ][
            local_index
        ]

        * segment[
            "highs_gain"
        ]

        * intensidad_agudos

        * lateral_window
    )


    ws[
        :,
        middle_end:
    ] += (

        highs_direction

        * float(
            highs_amount
        )
    )


    # ========================================================
    # NOISE
    # ========================================================

    noise_factor = float(

        noise_factor_curve[
            frame_index
        ]
    )


    # ========================================================
    # SÍNTESIS
    # ========================================================

    image = synthesize_dynamic(

        ws,

        noise_factor
    )


    # ========================================================
    # TENSOR -> RGB
    # ========================================================

    image = (

        (
            image
            .clamp(
                -1,
                1
            )
            + 1
        )

        * 127.5
    )


    image = (

        image

        .permute(
            0,
            2,
            3,
            1
        )

        .cpu()

        .numpy()

        .astype(
            np.uint8
        )
    )


    return image[0]


# ============================================================
# PRIMER FRAME
# ============================================================

first_frame = generate_audio_frame(
    0
)


frame_height = (
    first_frame.shape[0]
)


frame_width = (
    first_frame.shape[1]
)


print()

print(
    f"Resolución: "
    f"{frame_width} × {frame_height}"
)


if (

    frame_width % 2 != 0

    or

    frame_height % 2 != 0
):

    raise ValueError(
        "Ancho y alto deben ser pares."
    )


# ============================================================
# VIDEO TEMPORAL
# ============================================================

temp_video = (
    "/content/"
    "stylegan_audio_reactive_silent.mp4"
)


output_folder = os.path.dirname(
    video_path
)


if output_folder:

    os.makedirs(
        output_folder,
        exist_ok=True
    )


print()

print(
    "Generando frames..."
)


writer = imageio.get_writer(

    temp_video,

    fps=fps_video,

    codec="libx264",

    quality=8,

    output_params=[
        "-pix_fmt",
        "yuv420p"
    ]
)


try:

    writer.append_data(
        first_frame
    )


    for frame_index in tqdm(

        range(
            1,
            total_frames_video
        ),

        total=(
            total_frames_video
            - 1
        ),

        desc="Frames"
    ):

        frame = generate_audio_frame(
            frame_index
        )


        writer.append_data(
            frame
        )


finally:

    writer.close()


# ============================================================
# RESTAURAR NOISE
# ============================================================

if len(
    noise_parameters
) > 0:

    with torch.no_grad():

        for item in noise_parameters:

            item[
                "parameter"
            ].copy_(

                item[
                    "base"
                ]
            )


# ============================================================
# AÑADIR AUDIO
# ============================================================

print()

print(
    "Añadiendo audio..."
)


duracion_video = (

    total_frames_video
    / fps_video
)


duracion_mux = min(

    audio_segment_duration,

    duracion_video
)


ffmpeg_command = [

    "ffmpeg",

    "-y",

    "-loglevel",
    "error",

    "-i",
    temp_video,

    "-ss",
    f"{audio_offset:.6f}",

    "-i",
    str(
        AUDIO_PATH
    ),

    "-map",
    "0:v:0",

    "-map",
    "1:a:0",

    "-t",
    f"{duracion_mux:.6f}",

    "-c:v",
    "copy",

    "-c:a",
    "aac",

    "-b:a",
    "192k",

    "-shortest",

    video_path
]


result = subprocess.run(

    ffmpeg_command,

    text=True,

    capture_output=True
)


if result.returncode != 0:

    print(
        result.stderr
    )

    raise RuntimeError(
        "FFmpeg no pudo crear el video final."
    )


# ============================================================
# RESULTADO
# ============================================================

print()

print(
    "Video generado:"
)

print(
    video_path
)


print()

print(
    "Vista previa:"
)


display(

    Video(

        video_path,

        embed=True,

        html_attributes=(
            "controls "
            "playsinline "
            "style='max-width:900px; width:100%;'"
        )
    )
)